# Pawnee National Grassland Land Swap
## Compositing ecological, economic, contiguous, oil and gas, and connectivity metrics into a parcel matrix
- **Objective:** In this notebook, we use parcel data and the interior edge ratio metrics calculated in `07_contiguous_area.ipynb` to identify land swaps that would increase the interior edge ratio value. 

A future objective of this notebook is to incorporate the spatial metrics for species, land value, connectivity value, and oil and gas into consideration for the land swaps. The matrix should consider the total value from the metrics to identify best swaps. 

- **Author:** Max Warnock
- **Code review and/or edits:** 
- **Date:** June 17, 2026
- **Last date of revision:** June 19, 2026

---
### 🛠️ Prerequisites & Setup
* **Libraries:** 
* **Environment:**
* **Data Sources:** 
* **Related Notebooks:** 
* **Notes:**

### 🏗️ Methodology
1. Classify parcel ownership (federal, state, private) and project all parcel data to EPSG:5070 for metric calculations.
2. Build a rook-neighbor adjacency graph for all federal parcels and parse patches into NetworkX graphs; identify articulation points that must not be released.
3. Select receive-candidate patches using a variance-based score (`area_pct × ratio_pct × (1 - ratio_pct)`), which favors large patches with mid-range consolidation. Large, highly consolidated patches (above `SIZE_FLOOR_ACRES` and `SUBPATCH_RATIO_FLOOR`) are excluded from whole-patch analysis and handled exclusively by sub-patch decomposition.
4. For each receive candidate, find all adjacent non-federal parcels as acquisition candidates.
5. Evaluate one-to-one swap proposals: for each acquire candidate, find a releasable federal parcel within proximity and area tolerance, compute the hypothetical new `interior_edge_ratio`, and retain proposals with a positive net gain.
6. For large consolidated patches, decompose at articulation points into sub-clusters and run a separate swap evaluation scored against each sub-cluster's own ratio. Additional boundary checks prevent interior holes and enclave-shifting in the parent patch.
7. Rank proposals by net gain and visualize the top results on an interactive map.

---
### Reproducibility Notes

---
### ⚡ Troubleshooting/Notes
* 

---

# Libraries

In [1]:
### file paths, OS operations, utilities
import os
import pathlib

### geodata
import geopandas as gpd
from shapely.ops import unary_union

### geospatial plotting
import holoviews as hv
import geoviews as gv
import hvplot.pandas
from cartopy import crs as ccrs

### data handling
import pandas as pd
from collections import defaultdict

### graph analysis
import networkx as nx

### Jupyter display
from IPython.display import display

# Primary Directory

In [2]:
### set up root file path
# Walk up from the current directory to find the repo root (contains .git)
_cwd = pathlib.Path(os.getcwd()).resolve()
repo_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / '.git').exists()),
    _cwd
)
os.chdir(repo_root)

data_dir = os.path.join(repo_root, 'data')
os.makedirs(data_dir, exist_ok=True)

print(f'Repo root: {repo_root}')

Repo root: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project


# Secondary Directories

In [ ]:
### boundary dirs
boundary_dir       = os.path.join(data_dir, 'boundaries')
boundary_dir_final = os.path.join(boundary_dir, 'boundary-data-final')


### parcel data
parcel_bound      = os.path.join(boundary_dir_final, 'parcel_boundary')
parcel_bound_path = os.path.join(parcel_bound, 'pawnee_parcel.shp')
parcel_bound_gdf  = gpd.read_file(parcel_bound_path)


### output directories
contiguous_dir            = os.path.join(data_dir, 'contiguous')
figures_dir               = os.path.join(repo_root, 'figures')
figures_parcel_matrix_dir = os.path.join(figures_dir, 'parcel_matrix')
os.makedirs(contiguous_dir, exist_ok=True)
os.makedirs(figures_parcel_matrix_dir, exist_ok=True)


### load federal patches from 07_contiguous_area.ipynb output
federal_patches_path = os.path.join(contiguous_dir, 'federal_patches.gpkg')
federal_patches_gdf  = gpd.read_file(federal_patches_path)


print(f'Parcel data loaded : {len(parcel_bound_gdf)} parcels')
print(f'Federal patches    : {len(federal_patches_gdf)} patches')

# Classify Ownership

In [4]:
### classify ownership of parcels
def classify_ownership(name):
    if name == 'U S A':
        return 'FEDERAL'
    elif name == 'COLORADO STATE OF':
        return 'STATE'
    else:
        return 'PRIVATE'

### apply to the parcel_bound_gdf
parcel_bound_gdf['ownership'] = parcel_bound_gdf['NAME'].apply(classify_ownership)

# Land Swap Analysis
We want to identify federal-to-non-federal (state or private) land swap proposals that increase the `interior_edge_ratio` of federal patches.

**Main swap analysis:**
- **Receive candidates** — federal patches scored by `area_pct × ratio_pct × (1 - ratio_pct)`, which peaks at mid-range consolidation. Large patches that are too consolidated for whole-patch swaps (`interior_edge_ratio ≥ SUBPATCH_RATIO_FLOOR`) are excluded here and handled by sub-patch analysis instead.
- **Acquire candidates** — any non-federal parcel (state or private) sharing a rook edge with a receive patch.
- **Release candidates** — any federal parcel (from any patch) that is not an articulation point, has its centroid within `PROXIMITY_RADIUS_M` of the acquire parcel's centroid, and has area within `AREA_TOLERANCE` of the acquire parcel's area.

**Sub-patch analysis:**
Large, highly consolidated patches are decomposed at articulation points into sub-clusters. Each sub-cluster is evaluated independently against its own sub-patch ratio. Release candidates are restricted to outer-boundary parcels to avoid punching holes in the parent patch.

Proposals are one-to-one (one federal parcel traded for one non-federal parcel) and ranked by the net gain in `interior_edge_ratio`.

## Analysis Parameters

Key thresholds that control which swap proposals are generated. We can adjust these values to broaden or narrow the candidate pool:

- **`PROXIMITY_RADIUS_M`** — Maximum centroid-to-centroid distance (meters) between the acquired non-federal parcel and the released federal parcel. Ensures swaps are geographically paired.
- **`AREA_TOLERANCE`** — Hard cutoff: pairs where acreage differs by more than this fraction are skipped entirely.
- **`AREA_FLAG`** — Soft warning: proposals where acreage differs by more than this fraction are flagged in the summary for review.
- **`N_RECEIVE`** — Number of top-scored receive-candidate patches to evaluate as acquisition targets.
- **`MIN_SHARED_M`** — Minimum shared boundary length (meters) required for two parcels to be counted as rook neighbors.
- **`SIZE_FLOOR_ACRES`** — Any patch above this acreage is always included as a receive candidate, regardless of its receive score. Patches above this threshold are also candidates for sub-patch analysis.
- **`MIN_SUBPATCH_ACRES`** — In sub-patch analysis, an articulation point is only used as a cut point if every resulting component is at least this large. Ensures sub-patches are ecologically meaningful rather than tiny fragments.
- **`SUBPATCH_RATIO_FLOOR`** — Sub-patch analysis only applies to patches whose `interior_edge_ratio` is at or above this value. Patches below this threshold are already good main swap candidates and are kept in the main analysis only, avoiding proposal overlaps between the two analyses.
- **`SUBPATCH_PROXIMITY_RADIUS_M`** — Proximity radius used in sub-patch analysis instead of `PROXIMITY_RADIUS_M`. Set larger than the main radius because sub-patch swaps often pair interior hole-fill acquires with distant outer-boundary releases within the same large patch.

In [5]:
### swap analysis parameters
PROXIMITY_RADIUS_M          = 5000   # 5 km max centroid-to-centroid distance between acquired and released parcel
AREA_TOLERANCE              = 0.10   # hard cutoff — skip if parcels differ > 10% in area
AREA_FLAG                   = 0.05   # soft flag  — warn in summary if parcels differ > 5% in area
N_RECEIVE                   = 10     # number of top receive-candidate patches to evaluate
MIN_SHARED_M                = 1.0    # minimum shared boundary length (m) to count as adjacent
SIZE_FLOOR_ACRES            = 10000  # always include patches above this acreage regardless of score
MIN_SUBPATCH_ACRES          = 3000   # min component size (ac) for a cut point to qualify in sub-patch analysis
SUBPATCH_RATIO_FLOOR        = 0.65   # sub-patch analysis only applies to patches with ratio >= this value
SUBPATCH_PROXIMITY_RADIUS_M = 8000   # larger radius for sub-patch swaps — interior holes pair with distant outer-boundary releases

## Project Data to Metric CRS

We are performing all swap geometry calculations in **EPSG:5070** (NAD83 / Conus Albers), which is an equal-area projection for the continental US. This ensures area comparisons and distance measurements are in meters and acres rather than degrees.

Federal and non-federal parcels are separated into distinct GeoDataFrames, and a geometry lookup dictionary keyed by `PARCEL` ID is built for fast inner-loop access during swap evaluation.

In [6]:
### project all parcel data to EPSG:5070 for swap analysis
parcels_proj = parcel_bound_gdf.to_crs(epsg=5070).copy()
federal_proj  = parcels_proj[parcels_proj['ownership'] == 'FEDERAL'].copy().reset_index(drop=True)
nonfed_proj   = parcels_proj[parcels_proj['ownership'] != 'FEDERAL'].copy().reset_index(drop=True)

### geometry lookup for federal parcels, keyed by PARCEL string
parcel_geom_lookup = federal_proj.set_index('PARCEL')['geometry'].to_dict()

print(f"Federal parcels                       : {len(federal_proj)}")
print(f"Non-federal parcels (state + private) : {len(nonfed_proj)}")

Federal parcels                       : 598
Non-federal parcels (state + private) : 2876


## Build a boundary edge graph for neighboring parcels

Identifies all pairs of federal parcels that share a meaningful boundary edge (rook adjacency). A spatial index narrows the candidate pairs before computing the exact boundary intersection via Shapely.

Only `LineString` or `MultiLineString` intersections count as shared edges — corner-only `Point` touches are ignored. Results are stored as a list of `(PARCEL_i, PARCEL_j, shared_length_m)` tuples used to build per-patch graphs in the next step.

In [7]:
### build rook-neighbor edge list for all federal parcels
### uses spatial index to narrow candidates before boundary intersection check
### edges: list of (PARCEL_i, PARCEL_j, shared_length_m)

sindex_fed = federal_proj.sindex
fed_edges  = []

for i, row_i in federal_proj.iterrows():
    candidates = list(sindex_fed.intersection(row_i.geometry.bounds))
    for j in candidates:
        if j <= i:
            continue
        shared = row_i.geometry.boundary.intersection(federal_proj.geometry[j].boundary)
        if shared.is_empty:
            continue
        if shared.geom_type in ('LineString', 'MultiLineString'):
            length = shared.length
        elif shared.geom_type == 'GeometryCollection':
            lines  = [g for g in shared.geoms if g.geom_type in ('LineString', 'MultiLineString')]
            length = sum(g.length for g in lines)
        else:
            continue
        if length >= MIN_SHARED_M:
            fed_edges.append((federal_proj.loc[i, 'PARCEL'], federal_proj.loc[j, 'PARCEL'], length))

print(f"Found {len(fed_edges)} federal rook-neighbor pairs among {len(federal_proj)} parcels.")

Found 598 federal rook-neighbor pairs among 598 parcels.


## Parse Patch Structure & Build NetworkX Graphs

Each federal patch is a group of parcels that physically touch each other. This step figures out which parcels belong to which patch, then maps out the connections between them as a graph, where each parcel is a node and each shared boundary is an edge.

Once we have that graph, we can identify **articulation points**: parcels that are acting as a "bridge" holding two parts of a patch together. If you removed one of these parcels, the patch would split into two disconnected pieces. We flag these so the swap algorithm never proposes releasing one.

In [8]:
### parse patch membership from federal_patches_gdf
patch_parcel_sets = {}   # patch_id -> set of PARCEL strings
parcel_to_patch   = {}   # PARCEL string -> patch_id

for _, row in federal_patches_gdf.iterrows():
    pid         = row['contig_parcel_id']
    parcels_val = row['parcels']
    if not isinstance(parcels_val, str):
        continue   # skip patches where parcels column is NaN
    parcels = {p.strip() for p in parcels_val.split(',')}
    patch_parcel_sets[pid] = parcels
    for p in parcels:
        parcel_to_patch[p] = pid

### per-patch edge lists and interior sums
patch_edges = defaultdict(list)
for p1, p2, length in fed_edges:
    patch = parcel_to_patch.get(p1)
    if patch and parcel_to_patch.get(p2) == patch:
        patch_edges[patch].append((p1, p2, length))

patch_interior_sum = {
    pid: sum(l for _, _, l in edges)
    for pid, edges in patch_edges.items()
}

### per-patch networkx graphs and articulation points
patch_graphs     = {}
patch_art_points = {}
for pid, parcel_ids in patch_parcel_sets.items():
    G = nx.Graph()
    G.add_nodes_from(parcel_ids)
    for p1, p2, _ in patch_edges[pid]:
        G.add_edge(p1, p2)
    patch_graphs[pid]     = G
    patch_art_points[pid] = set(nx.articulation_points(G))

n_art = sum(len(v) for v in patch_art_points.values())
print(f"Parsed {len(patch_parcel_sets)} patches — {n_art} articulation points identified.")

Parsed 148 patches — 133 articulation points identified.


## Select Target Patches for Growth

Not all federal patches are worth trying to grow. We want to focus on patches that are **large and partially consolidated** — big enough to matter ecologically, already structured enough that swaps can meaningfully improve them, but not so consolidated that there's little room left to gain.

The receive score is `area_pct × ratio_pct × (1 - ratio_pct)`, which peaks at ratio_pct = 0.5 and drops to zero at both extremes. A completely fragmented patch (ratio ≈ 0) and a fully consolidated patch (ratio ≈ 1) both score near zero; the highest scores go to large patches in the middle range that have meaningful structure and meaningful room to improve.

The top `N_RECEIVE` patches by this score are selected. Additionally, any patch exceeding `SIZE_FLOOR_ACRES` with `interior_edge_ratio < SUBPATCH_RATIO_FLOOR` is always included regardless of its score. Large patches at or above `SUBPATCH_RATIO_FLOOR` (e.g., PATCH_001, PATCH_002) are intentionally excluded here — they are already too consolidated for whole-patch swaps to help and are instead analyzed exclusively via sub-patch decomposition in the next section.

In [9]:
### select receive candidate patches
### score = area_pct × ratio_pct × (1 - ratio_pct): large patches with mid-range ratio
### size floor: always include patches above SIZE_FLOOR_ACRES that are NOT handled by sub-patch analysis
### patches above SIZE_FLOOR_ACRES with ratio >= SUBPATCH_RATIO_FLOOR are excluded here
### and analyzed exclusively via sub-patch decomposition in the next section

patches_multi = federal_patches_gdf[federal_patches_gdf['n_parcels'] > 1].copy()
patches_multi['_area_pct']      = patches_multi['area_acres'].rank(pct=True)
patches_multi['_ratio_pct']     = patches_multi['interior_edge_ratio'].rank(pct=True)
patches_multi['_receive_score'] = patches_multi['_area_pct'] * patches_multi['_ratio_pct'] * (1 - patches_multi['_ratio_pct'])

scored = patches_multi.nlargest(N_RECEIVE, '_receive_score')
large  = patches_multi[
    (patches_multi['area_acres'] >= SIZE_FLOOR_ACRES) &
    (patches_multi['interior_edge_ratio'] < SUBPATCH_RATIO_FLOOR) &
    (~patches_multi['contig_parcel_id'].isin(scored['contig_parcel_id']))
]
receive_candidates = pd.concat([scored, large]).reset_index(drop=True)

print(f"Top {N_RECEIVE} scored + {len(large)} size-floor patches = {len(receive_candidates)} receive candidates:")
print(
    receive_candidates[['contig_parcel_id', 'area_acres', 'n_parcels', 'interior_edge_ratio']]
    .to_string(index=False)
)

Top 10 scored + 1 size-floor patches = 11 receive candidates:
contig_parcel_id   area_acres  n_parcels  interior_edge_ratio
       PATCH_011  4026.737934       10.0               0.3464
       PATCH_015  2711.659739        5.0               0.3007
       PATCH_016  2654.165243        6.0               0.2205
       PATCH_010  4309.266388       11.0               0.4260
       PATCH_005  8032.243481       15.0               0.5092
       PATCH_004 14462.476168       28.0               0.5295
       PATCH_012  3325.158556        8.0               0.4990
       PATCH_020  1965.403303        3.0               0.2060
       PATCH_019  2207.206276        5.0               0.3871
       PATCH_022  1440.720785        5.0               0.2792
       PATCH_003 15766.553980       54.0               0.6465


## Identify Adjacent Non-Federal Parcels

For each receive-candidate patch, finds all non-federal (state or private) parcels that share a rook edge with any parcel in the patch. These are the candidate parcels to **acquire** — adding one of these to a federal patch increases its area and, if well-positioned, its interior edge ratio.

In [10]:
### find non-federal parcels sharing a rook edge with any parcel in each receive candidate patch
### adj_edges: (patch_id, nf_idx) -> list of (fed_parcel_id, shared_length_m)

sindex_nonfed = nonfed_proj.sindex
adj_edges     = defaultdict(list)

for patch_id in receive_candidates['contig_parcel_id']:
    for fed_parcel_id in patch_parcel_sets[patch_id]:
        fed_geom   = parcel_geom_lookup[fed_parcel_id]
        candidates = list(sindex_nonfed.intersection(fed_geom.bounds))
        for nf_idx in candidates:
            nf_geom = nonfed_proj.geometry[nf_idx]
            shared  = fed_geom.boundary.intersection(nf_geom.boundary)
            if shared.is_empty:
                continue
            if shared.geom_type in ('LineString', 'MultiLineString'):
                length = shared.length
            elif shared.geom_type == 'GeometryCollection':
                lines  = [g for g in shared.geoms if g.geom_type in ('LineString', 'MultiLineString')]
                length = sum(g.length for g in lines)
            else:
                continue
            if length >= MIN_SHARED_M:
                adj_edges[(patch_id, nf_idx)].append((fed_parcel_id, length))

n_unique_nf = len({nf_idx for _, nf_idx in adj_edges})
print(f"Found {len(adj_edges)} (patch, non-federal parcel) adjacent pairs "
      f"({n_unique_nf} unique non-federal parcels).")

Found 270 (patch, non-federal parcel) adjacent pairs (258 unique non-federal parcels).


## Evaluate Swap Proposals

For each (receive patch, adjacent non-federal parcel) pair, searches for a releasable federal parcel to trade in return. A federal parcel qualifies as a release candidate if:

1. It is **not** an articulation point in its own patch (removing it would not fragment that patch)
2. Its centroid is within `PROXIMITY_RADIUS_M` of the acquired parcel's centroid (geographically paired)
3. Its area is within `AREA_TOLERANCE` of the acquired parcel's area (roughly equal exchange)

The new `interior_edge_ratio` for the receive patch is computed after the hypothetical swap. Only proposals where the ratio **increases** are retained. Release patch impacts are cached to avoid redundant dissolve operations on the same (patch, parcel) combination.

In [11]:
### evaluate swap proposals
### for each receive patch + adjacent non-federal parcel, find releasable federal parcels
### release parcel may come from any patch (cross-patch swap) as long as:
###   1. not an articulation point in its own patch
###   2. centroid within PROXIMITY_RADIUS_M of the non-federal parcel's centroid
###   3. area within AREA_TOLERANCE of the non-federal parcel's area

### precompute dissolved patch geometry and old ratio per patch
patch_dissolved_geom = {
    row['contig_parcel_id']: row['geometry']
    for _, row in federal_patches_gdf.iterrows()
}
patch_old_ratio = {
    row['contig_parcel_id']: row['interior_edge_ratio']
    for _, row in federal_patches_gdf.iterrows()
}

### federal parcel info dict for fast inner-loop access
fed_info = {
    row['PARCEL']: {
        'geom':     row['geometry'],
        'area':     row['geometry'].area,
        'centroid': row['geometry'].centroid,
        'patch_id': parcel_to_patch.get(row['PARCEL']),
    }
    for _, row in federal_proj.iterrows()
}

### spatial index on federal parcel centroids for proximity pre-filter
fed_centroid_gdf             = federal_proj.copy()
fed_centroid_gdf['geometry'] = federal_proj.geometry.centroid
fed_centroid_sindex          = fed_centroid_gdf.sindex

### cache: (release_patch_id, release_parcel_id) -> new_ratio after losing that parcel
release_cache = {}

proposals = []
print("Evaluating swap proposals...")

for (patch_id, nf_idx), edges_to_patch in adj_edges.items():
    nf_geom         = nonfed_proj.geometry[nf_idx]
    nf_area         = nf_geom.area
    nf_centroid     = nf_geom.centroid
    nf_acreage      = nf_area * 0.000247105
    old_ratio       = patch_old_ratio[patch_id]
    old_interior    = patch_interior_sum.get(patch_id, 0.0)
    new_cross_edges = sum(l for _, l in edges_to_patch)

    ### precompute receive patch + nf for cross-patch scenario (two-geometry union, fast)
    recv_with_nf_geom      = patch_dissolved_geom[patch_id].union(nf_geom)
    recv_with_nf_perimeter = recv_with_nf_geom.length
    recv_with_nf_interior  = old_interior + new_cross_edges

    ### precompute augmented graph art points for same-patch swap checks (once per nf parcel)
    G_aug = patch_graphs[patch_id].copy()
    G_aug.add_node('_nf_')
    for fp, _ in edges_to_patch:
        G_aug.add_edge('_nf_', fp)
    aug_art_points = set(nx.articulation_points(G_aug))

    ### proximity pre-filter: federal centroids within PROXIMITY_RADIUS_M
    search_buffer = nf_centroid.buffer(PROXIMITY_RADIUS_M)
    nearby_rows   = list(fed_centroid_sindex.intersection(search_buffer.bounds))

    for row_idx in nearby_rows:
        f_parcel_id = federal_proj.loc[row_idx, 'PARCEL']
        f           = fed_info[f_parcel_id]
        f_patch_id  = f['patch_id']
        if f_patch_id is None:
            continue

        ### exact distance check
        if nf_centroid.distance(f['centroid']) > PROXIMITY_RADIUS_M:
            continue

        ### area constraint
        area_diff = abs(nf_area - f['area']) / max(nf_area, f['area'])
        if area_diff > AREA_TOLERANCE:
            continue

        same_patch = (f_patch_id == patch_id)

        if same_patch:
            ### articulation check in augmented graph (nf parcel already added)
            if f_parcel_id in aug_art_points:
                continue

            ### new interior sum: remove f_parcel_id edges, add nf cross edges
            lost         = sum(l for p1, p2, l in patch_edges[patch_id]
                               if p1 == f_parcel_id or p2 == f_parcel_id)
            new_interior = old_interior - lost + new_cross_edges

            ### new perimeter: dissolve remaining parcels + nf parcel
            remaining_geoms = (
                [parcel_geom_lookup[p] for p in patch_parcel_sets[patch_id] if p != f_parcel_id]
                + [nf_geom]
            )
            new_perimeter     = unary_union(remaining_geoms).length
            new_ratio         = new_interior / new_perimeter if new_perimeter > 0 else 0.0
            release_old_ratio = old_ratio
            release_new_ratio = new_ratio   # same patch — receive and release are one operation

        else:
            ### cross-patch: receive patch gains nf parcel; check art points in release patch
            if f_parcel_id in patch_art_points.get(f_patch_id, set()):
                continue
            new_ratio = recv_with_nf_interior / recv_with_nf_perimeter if recv_with_nf_perimeter > 0 else 0.0

            ### release patch impact (cached to avoid repeated dissolves)
            cache_key = (f_patch_id, f_parcel_id)
            if cache_key not in release_cache:
                rel_old_interior = patch_interior_sum.get(f_patch_id, 0.0)
                lost             = sum(l for p1, p2, l in patch_edges[f_patch_id]
                                       if p1 == f_parcel_id or p2 == f_parcel_id)
                rel_remaining    = [parcel_geom_lookup[p]
                                    for p in patch_parcel_sets[f_patch_id] if p != f_parcel_id]
                if rel_remaining:
                    rel_perimeter            = unary_union(rel_remaining).length
                    release_cache[cache_key] = (rel_old_interior - lost) / rel_perimeter if rel_perimeter > 0 else 0.0
                else:
                    release_cache[cache_key] = 0.0   # single-parcel patch fully traded away

            release_old_ratio = patch_old_ratio[f_patch_id]
            release_new_ratio = release_cache[cache_key]

        net_gain = new_ratio - old_ratio
        if net_gain <= 0:
            continue

        proposals.append(dict(
            receive_patch_id  = patch_id,
            old_ratio         = round(old_ratio, 4),
            new_ratio         = round(new_ratio, 4),
            net_gain          = round(net_gain, 4),
            nf_parcel_id      = nonfed_proj.loc[nf_idx, 'PARCEL'],
            nf_ownership      = nonfed_proj.loc[nf_idx, 'ownership'],
            nf_acres          = round(nf_acreage, 1),
            release_parcel_id = f_parcel_id,
            release_patch_id  = f_patch_id,
            fed_acres         = round(f['area'] * 0.000247105, 1),
            area_diff_pct     = round(area_diff * 100, 1),
            area_flag         = area_diff > AREA_FLAG,
            distance_km       = round(nf_centroid.distance(f['centroid']) / 1000, 2),
            same_patch        = same_patch,
            release_old_ratio = round(release_old_ratio, 4),
            release_new_ratio = round(release_new_ratio, 4),
        ))

print(f"Found {len(proposals)} valid swap proposals.")

Evaluating swap proposals...


Found 227 valid swap proposals.


## Results Summary

Proposals are ranked by `net_gain` — the increase in `interior_edge_ratio` for the receive patch. Each proposal reports:

- Which patch gains land and how its ratio improves
- The non-federal parcel being acquired (ownership type, acreage)
- The federal parcel being released (acreage, source patch)
- Acreage difference and centroid-to-centroid distance between the two parcels
- Release patch ratio before and after (cross-patch swaps only)

In [12]:
### print ranked summary of swap proposals

if not proposals:
    print("No valid swap proposals found.")
else:
    proposals_df = (
        pd.DataFrame(proposals)
        .sort_values('net_gain', ascending=False)
        .reset_index(drop=True)
    )
    proposals_df.index += 1   # 1-based rank

    print("=" * 70)
    print("  LAND SWAP PROPOSAL SUMMARY")
    print("=" * 70)

    for rank, row in proposals_df.head(30).iterrows():
        flags = []
        if row['area_flag']:
            flags.append(
                f"AREA DIFFERENCE {row['area_diff_pct']}%  "
                f"(acquire {row['nf_acres']} ac / release {row['fed_acres']} ac)"
            )
        flag_str   = ("\n    [!]  " + "\n    [!]  ".join(flags)) if flags else ""
        swap_label = "same-patch" if row['same_patch'] else "cross-patch"

        print(f"\n  Proposal #{rank}  |  {row['receive_patch_id']}  [{swap_label}]")
        print(f"    Receive patch ratio : {row['old_ratio']:.4f}  ->  {row['new_ratio']:.4f}  (+{row['net_gain']:.4f})")
        print(f"    Acquire  : {row['nf_ownership']} parcel  {row['nf_parcel_id']}  ({row['nf_acres']} ac)")
        print(f"    Release  : FEDERAL parcel  {row['release_parcel_id']}  ({row['fed_acres']} ac)  "
              f"from {row['release_patch_id']}")
        acreage_diff = abs(row['nf_acres'] - row['fed_acres'])
        print(f"    Area difference     : {row['area_diff_pct']}%  ({acreage_diff:.1f} ac)")
        print(f"    Distance            : {row['distance_km']} km")
        if not row['same_patch']:
            print(f"    Release patch ratio : {row['release_old_ratio']:.4f}  ->  {row['release_new_ratio']:.4f}")
        if flag_str:
            print(flag_str)

    print("\n" + "=" * 70)
    print(f"\nTotal proposals found : {len(proposals_df)}")
    print(f"Shown above           : top {min(30, len(proposals_df))}")

  LAND SWAP PROPOSAL SUMMARY

  Proposal #1  |  PATCH_019  [same-patch]
    Receive patch ratio : 0.3871  ->  0.6406  (+0.2535)
    Acquire  : PRIVATE parcel  003126000001  (319.1 ac)
    Release  : FEDERAL parcel  022103000005  (321.8 ac)  from PATCH_019
    Area difference     : 0.8%  (2.7 ac)
    Distance            : 3.86 km

  Proposal #2  |  PATCH_005  [cross-patch]
    Receive patch ratio : 0.5092  ->  0.6936  (+0.1844)
    Acquire  : PRIVATE parcel  020526000001  (320.2 ac)
    Release  : FEDERAL parcel  020719000004  (337.6 ac)  from PATCH_055
    Area difference     : 5.2%  (17.4 ac)
    Distance            : 3.63 km
    Release patch ratio : 0.0000  ->  0.0000

    [!]  AREA DIFFERENCE 5.2%  (acquire 320.2 ac / release 337.6 ac)

  Proposal #3  |  PATCH_015  [cross-patch]
    Receive patch ratio : 0.3007  ->  0.4716  (+0.1709)
    Acquire  : STATE parcel  003536000005  (621.2 ac)
    Release  : FEDERAL parcel  021908000003  (602.1 ac)  from PATCH_006
    Area difference     

## Sub-Patch Swap Analysis

Large, consolidated federal patches (above `SIZE_FLOOR_ACRES`) rarely produce valid proposals in the main swap evaluation because their high `interior_edge_ratio` leaves little room for whole-patch gains. However, these patches often contain internal sub-clusters connected by narrow corridors — articulation points that bridge otherwise-separate groups of parcels.

This step temporarily removes articulation points that divide a large patch into sub-clusters of at least `MIN_SUBPATCH_ACRES` each, treating each sub-cluster as an independent receive candidate. Swap proposals are evaluated against the **sub-patch ratio** (not the whole-patch ratio), revealing local consolidation opportunities that the global metric obscures.

Results are kept separate from the main swap analysis and displayed on the map in distinct colors.

In [13]:
### sub-patch swap analysis for large consolidated patches
### patches above SIZE_FLOOR_ACRES are decomposed at articulation points that split
### them into components >= MIN_SUBPATCH_ACRES; each component is treated as an
### independent receive candidate scored against its own sub-patch ratio
###
### SUBPATCH_RATIO_FLOOR restricts sub-patch analysis to patches already too consolidated
### for the main swap analysis to help, keeping the two analyses structurally separate

_parcel_ac = {pid: info['area'] * 0.000247105 for pid, info in fed_info.items()}

### pre-compute federal neighbor count per parcel from the global fed_edges list
### used to block release candidates that are too interior (3+ federal neighbors)
_fed_neighbor_count = {}
for _p1, _p2, _ in fed_edges:
    _fed_neighbor_count[_p1] = _fed_neighbor_count.get(_p1, 0) + 1
    _fed_neighbor_count[_p2] = _fed_neighbor_count.get(_p2, 0) + 1

large_patch_ids = federal_patches_gdf.loc[
    (federal_patches_gdf['area_acres'] >= SIZE_FLOOR_ACRES) &
    (federal_patches_gdf['interior_edge_ratio'] >= SUBPATCH_RATIO_FLOOR),
    'contig_parcel_id'
].tolist()

print(f"Sub-patch candidates (>= {SIZE_FLOOR_ACRES} ac, ratio >= {SUBPATCH_RATIO_FLOOR}): {large_patch_ids}\n")

subpatch_proposals = []

for patch_id in large_patch_ids:
    G = patch_graphs[patch_id]

    ### articulation points that split the patch into only large-enough components
    useful_cuts = set()
    for ap in list(nx.articulation_points(G)):
        G_test = G.copy()
        G_test.remove_node(ap)
        if min(
            sum(_parcel_ac.get(p, 0) for p in comp)
            for comp in nx.connected_components(G_test)
        ) >= MIN_SUBPATCH_ACRES:
            useful_cuts.add(ap)

    if not useful_cuts:
        print(f"{patch_id}: no qualifying cut points (all splits create a component < {MIN_SUBPATCH_ACRES} ac)")
        continue

    ### remove all qualifying cut points simultaneously
    G_cut    = G.copy()
    G_cut.remove_nodes_from(useful_cuts)
    sub_comps = [
        comp for comp in nx.connected_components(G_cut)
        if sum(_parcel_ac.get(p, 0) for p in comp) >= MIN_SUBPATCH_ACRES
    ]
    print(f"{patch_id}: {len(useful_cuts)} cut points removed → {len(sub_comps)} sub-patches")

    for sub_idx, sub_parcels in enumerate(sub_comps):
        sub_id  = f"{patch_id}_S{sub_idx + 1}"
        sub_ac  = sum(_parcel_ac.get(p, 0) for p in sub_parcels)

        ### sub-patch interior edge sum (only edges wholly within sub_parcels)
        sub_int  = sum(
            l for p1, p2, l in patch_edges[patch_id]
            if p1 in sub_parcels and p2 in sub_parcels
        )
        sub_perim     = unary_union([parcel_geom_lookup[p] for p in sub_parcels]).length
        sub_old_ratio = sub_int / sub_perim if sub_perim > 0 else 0.0

        ### sub-patch graph and its own articulation points
        G_sub   = G.subgraph(sub_parcels).copy()
        sub_art = set(nx.articulation_points(G_sub))

        ### non-federal parcels adjacent to any parcel in this sub-patch
        sub_adj = defaultdict(list)   # nf_idx -> [(fed_parcel_id, shared_m)]
        for fp in sub_parcels:
            for nf_idx in sindex_nonfed.intersection(parcel_geom_lookup[fp].bounds):
                shared = parcel_geom_lookup[fp].boundary.intersection(nonfed_proj.geometry[nf_idx].boundary)
                if shared.is_empty:
                    continue
                if shared.geom_type in ('LineString', 'MultiLineString'):
                    length = shared.length
                elif shared.geom_type == 'GeometryCollection':
                    lines  = [g for g in shared.geoms if g.geom_type in ('LineString', 'MultiLineString')]
                    length = sum(g.length for g in lines)
                else:
                    continue
                if length >= MIN_SHARED_M:
                    sub_adj[nf_idx].append((fp, length))

        ### federal parcels in this sub-patch that touch non-federal land
        ### release candidates must be in this set — interior parcels (touching only federal
        ### land) would punch a hole in the parent patch at the whole-patch scale
        sub_boundary_feds = {fp for edges in sub_adj.values() for fp, _ in edges}

        ### evaluate one-to-one swaps within the sub-patch
        for nf_idx, edges_to_sub in sub_adj.items():
            nf_geom         = nonfed_proj.geometry[nf_idx]
            nf_area         = nf_geom.area
            nf_centroid     = nf_geom.centroid
            nf_acreage      = nf_area * 0.000247105
            new_cross_edges = sum(l for _, l in edges_to_sub)

            ### count all federal rook-neighbors of the acquire parcel (across all patches)
            ### if >= 3 sides are federal the parcel is substantially enclosed; releasing a
            ### directly adjacent federal parcel would just shift the enclave one space
            nf_all_fed_neighbors = set()
            for _fi in sindex_fed.intersection(nf_geom.bounds):
                _sh = nf_geom.boundary.intersection(federal_proj.geometry[_fi].boundary)
                if _sh.is_empty:
                    continue
                if _sh.geom_type in ('LineString', 'MultiLineString'):
                    nf_all_fed_neighbors.add(federal_proj.loc[_fi, 'PARCEL'])
                elif _sh.geom_type == 'GeometryCollection':
                    if any(g.geom_type in ('LineString', 'MultiLineString') for g in _sh.geoms):
                        nf_all_fed_neighbors.add(federal_proj.loc[_fi, 'PARCEL'])
            acquire_is_enclosed = len(nf_all_fed_neighbors) >= 3

            ### augmented graph to check if nf parcel would replace an articulation point
            G_aug = G_sub.copy()
            G_aug.add_node('_nf_')
            for fp, _ in edges_to_sub:
                G_aug.add_edge('_nf_', fp)
            aug_art = set(nx.articulation_points(G_aug))

            nearby = list(fed_centroid_sindex.intersection(nf_centroid.buffer(SUBPATCH_PROXIMITY_RADIUS_M).bounds))

            for row_idx in nearby:
                f_parcel_id = federal_proj.loc[row_idx, 'PARCEL']
                if f_parcel_id not in sub_parcels:
                    continue   # release must come from within the sub-patch
                if f_parcel_id not in sub_boundary_feds:
                    continue   # release must touch non-federal land (outer boundary only)
                if _fed_neighbor_count.get(f_parcel_id, 0) >= 3:
                    continue   # release touches 3+ federal parcels — too interior, would create a hole
                if acquire_is_enclosed and f_parcel_id in nf_all_fed_neighbors:
                    continue   # hole-shifting: releasing an adjacent parcel just moves the enclave
                if f_parcel_id in sub_art or f_parcel_id in aug_art:
                    continue

                f        = fed_info[f_parcel_id]
                dist     = nf_centroid.distance(f['centroid'])
                if dist > SUBPATCH_PROXIMITY_RADIUS_M:
                    continue

                area_diff = abs(nf_area - f['area']) / max(nf_area, f['area'])
                if area_diff > AREA_TOLERANCE:
                    continue

                lost         = sum(
                    l for p1, p2, l in patch_edges[patch_id]
                    if (p1 == f_parcel_id and p2 in sub_parcels) or
                       (p2 == f_parcel_id and p1 in sub_parcels)
                )
                new_interior = sub_int - lost + new_cross_edges
                remaining    = [parcel_geom_lookup[p] for p in sub_parcels if p != f_parcel_id] + [nf_geom]
                new_perim    = unary_union(remaining).length
                new_ratio    = new_interior / new_perim if new_perim > 0 else 0.0
                net_gain     = new_ratio - sub_old_ratio

                if net_gain <= 0:
                    continue

                subpatch_proposals.append(dict(
                    parent_patch_id   = patch_id,
                    sub_patch_id      = sub_id,
                    sub_patch_acres   = round(sub_ac, 1),
                    sub_old_ratio     = round(sub_old_ratio, 4),
                    sub_new_ratio     = round(new_ratio, 4),
                    net_gain          = round(net_gain, 4),
                    nf_parcel_id      = nonfed_proj.loc[nf_idx, 'PARCEL'],
                    nf_ownership      = nonfed_proj.loc[nf_idx, 'ownership'],
                    nf_acres          = round(nf_acreage, 1),
                    release_parcel_id = f_parcel_id,
                    fed_acres         = round(f['area'] * 0.000247105, 1),
                    area_diff_pct     = round(area_diff * 100, 1),
                    area_flag         = area_diff > AREA_FLAG,
                    distance_km       = round(dist / 1000, 2),
                ))

print(f"\nSub-patch proposals found: {len(subpatch_proposals)}")

Sub-patch candidates (>= 10000 ac, ratio >= 0.65): ['PATCH_001', 'PATCH_002']

PATCH_001: 13 cut points removed → 4 sub-patches
PATCH_002: no qualifying cut points (all splits create a component < 3000 ac)

Sub-patch proposals found: 48


## Sub-Patch Results Summary

In [14]:
### print ranked sub-patch swap summary

if not subpatch_proposals:
    print("No sub-patch proposals found.")
    subpatch_df = pd.DataFrame()
else:
    subpatch_df = (
        pd.DataFrame(subpatch_proposals)
        .sort_values('net_gain', ascending=False)
        .reset_index(drop=True)
    )
    subpatch_df.index += 1   # 1-based rank

    print("=" * 70)
    print("  SUB-PATCH SWAP PROPOSAL SUMMARY")
    print("=" * 70)

    for rank, row in subpatch_df.head(30).iterrows():
        flags = []
        if row['area_flag']:
            flags.append(
                f"AREA DIFFERENCE {row['area_diff_pct']}%  "
                f"(acquire {row['nf_acres']} ac / release {row['fed_acres']} ac)"
            )
        flag_str = ("\n    [!]  " + "\n    [!]  ".join(flags)) if flags else ""

        print(f"\n  Sub-Proposal #{rank}  |  {row['sub_patch_id']}  (parent: {row['parent_patch_id']})")
        print(f"    Sub-patch area      : {row['sub_patch_acres']} ac")
        print(f"    Sub-patch ratio     : {row['sub_old_ratio']:.4f}  ->  {row['sub_new_ratio']:.4f}  (+{row['net_gain']:.4f})")
        print(f"    Acquire  : {row['nf_ownership']} parcel  {row['nf_parcel_id']}  ({row['nf_acres']} ac)")
        print(f"    Release  : FEDERAL parcel  {row['release_parcel_id']}  ({row['fed_acres']} ac)")
        print(f"    Area difference     : {row['area_diff_pct']}%")
        print(f"    Distance            : {row['distance_km']} km")
        if flag_str:
            print(flag_str)

    print("\n" + "=" * 70)
    n_sub = subpatch_df['sub_patch_id'].nunique()
    n_par = subpatch_df['parent_patch_id'].nunique()
    print(f"\nTotal sub-patch proposals : {len(subpatch_df)}  across {n_sub} sub-patches  ({n_par} parent patches)")
    print(f"Shown above               : top {min(30, len(subpatch_df))}")

  SUB-PATCH SWAP PROPOSAL SUMMARY

  Sub-Proposal #1  |  PATCH_001_S4  (parent: PATCH_001)
    Sub-patch area      : 3724.5 ac
    Sub-patch ratio     : 0.3336  ->  0.5150  (+0.1815)
    Acquire  : STATE parcel  054505000006  (613.2 ac)
    Release  : FEDERAL parcel  054518000001  (643.9 ac)
    Area difference     : 4.8%
    Distance            : 3.61 km

  Sub-Proposal #2  |  PATCH_001_S1  (parent: PATCH_001)
    Sub-patch area      : 19015.1 ac
    Sub-patch ratio     : 0.6508  ->  0.7827  (+0.1319)
    Acquire  : PRIVATE parcel  046127000005  (640.5 ac)
    Release  : FEDERAL parcel  046115000004  (642.9 ac)
    Area difference     : 0.4%
    Distance            : 3.29 km

  Sub-Proposal #3  |  PATCH_001_S3  (parent: PATCH_001)
    Sub-patch area      : 4818.6 ac
    Sub-patch ratio     : 0.4422  ->  0.5579  (+0.1157)
    Acquire  : PRIVATE parcel  029729000009  (649.2 ac)
    Release  : FEDERAL parcel  029725000004  (620.3 ac)
    Area difference     : 4.4%
    Distance           

## Swap Map

Visualizes the top-ranked swap proposals on an interactive map. Pan, zoom, and hover over any parcel to see its metadata.

- **Amber patches** — federal contiguous patches (multi-parcel only)
- **Green parcels** — non-federal land proposed for acquisition (main swaps), labeled **A#**
- **Red parcels** — federal land proposed for release (main swaps), labeled **R#**
- **Blue parcels** — non-federal land proposed for acquisition (sub-patch swaps), labeled **SA#**
- **Orange parcels** — federal land proposed for release (sub-patch swaps), labeled **SR#**
- Parcels shared across multiple proposals carry a combined label (e.g., **A1/A3**)

The map is saved as a standalone interactive HTML file to `figures/parcel_matrix/`.

In [ ]:
if 'proposals_df' not in dir() or proposals_df.empty:
    print("No proposals to map.")
else:
    import geoviews.tile_sources as gvts
    from bokeh.models import GlyphRenderer, Legend, LegendItem, ColumnDataSource
    from bokeh.models.glyphs import MultiPolygons as _MP
    from bokeh.resources import CDN
    from bokeh.embed import file_html, components
    from IPython.display import HTML

    TOP_N          = 30
    ACQUIRE_COLOR  = '#2ca25f'
    RELEASE_COLOR  = '#d7301f'
    PATCH_COLOR    = '#fff29b'
    SUB_ACQ_COLOR  = '#2166ac'
    SUB_REL_COLOR  = '#f16913'

    plot_proposals = proposals_df.head(TOP_N)

    ### all polygon layers in EPSG:3857 (native tile CRS)
    ### gv.Polygons(crs=ccrs.GOOGLE_MERCATOR) declares input = display CRS so project_path
    ### performs an identity transform instead of the buggy ellipsoidal→spherical conversion
    ### that shifts 4326 polygons ~28 km north of the tile background at ~41°N
    parcels_3857 = parcel_bound_gdf.to_crs(epsg=3857)
    patches_3857 = (
        federal_patches_gdf[federal_patches_gdf['n_parcels'] > 1]
        .to_crs(epsg=3857)
        [['contig_parcel_id', 'area_acres', 'n_parcels', 'interior_edge_ratio', 'geometry']]
        .reset_index(drop=True)
    )

    nonfed_geom_idx = nonfed_proj.set_index('PARCEL')['geometry']

    ### build main acquire / release GDFs
    acq_5070 = gpd.GeoDataFrame(
        geometry=[nonfed_geom_idx[r['nf_parcel_id']] for _, r in plot_proposals.iterrows()],
        crs='EPSG:5070'
    )
    rel_5070 = gpd.GeoDataFrame(
        geometry=[parcel_geom_lookup[r['release_parcel_id']] for _, r in plot_proposals.iterrows()],
        crs='EPSG:5070'
    )
    acq_3857_all = acq_5070.to_crs(3857)
    rel_3857_all = rel_5070.to_crs(3857)

    seen_acq = {}
    for i, (rank, row) in enumerate(plot_proposals.iterrows()):
        pid = row['nf_parcel_id']
        cent = gpd.GeoSeries([acq_5070.geometry.iloc[i].centroid], crs=5070).to_crs(3857).iloc[0]
        if pid not in seen_acq:
            seen_acq[pid] = dict(
                geometry=acq_3857_all.geometry.iloc[i], label=f'A{rank}',
                label_x=cent.x, label_y=cent.y,
                ownership=row['nf_ownership'], acres=row['nf_acres'], proposals=str(rank),
            )
        else:
            seen_acq[pid]['label']     += f'/A{rank}'
            seen_acq[pid]['proposals'] += f', {rank}'

    seen_rel = {}
    for i, (rank, row) in enumerate(plot_proposals.iterrows()):
        pid = row['release_parcel_id']
        cent = gpd.GeoSeries([rel_5070.geometry.iloc[i].centroid], crs=5070).to_crs(3857).iloc[0]
        if pid not in seen_rel:
            seen_rel[pid] = dict(
                geometry=rel_3857_all.geometry.iloc[i], label=f'R{rank}',
                label_x=cent.x, label_y=cent.y,
                release_patch=row['release_patch_id'], acres=row['fed_acres'], proposals=str(rank),
            )
        else:
            seen_rel[pid]['label']     += f'/R{rank}'
            seen_rel[pid]['proposals'] += f', {rank}'

    acq_unique = gpd.GeoDataFrame(list(seen_acq.values()), geometry='geometry', crs='EPSG:3857')
    rel_unique = gpd.GeoDataFrame(list(seen_rel.values()), geometry='geometry', crs='EPSG:3857')

    ### build sub-patch acquire / release GDFs (if sub-patch analysis was run)
    _has_subpatch = 'subpatch_df' in dir() and not subpatch_df.empty
    if _has_subpatch:
        plot_sub  = subpatch_df.head(TOP_N)
        sacq_5070 = gpd.GeoDataFrame(
            geometry=[nonfed_geom_idx[r['nf_parcel_id']] for _, r in plot_sub.iterrows()],
            crs='EPSG:5070'
        )
        srel_5070 = gpd.GeoDataFrame(
            geometry=[parcel_geom_lookup[r['release_parcel_id']] for _, r in plot_sub.iterrows()],
            crs='EPSG:5070'
        )
        sacq_3857 = sacq_5070.to_crs(3857)
        srel_3857 = srel_5070.to_crs(3857)

        seen_sacq = {}
        for i, (rank, row) in enumerate(plot_sub.iterrows()):
            pid  = row['nf_parcel_id']
            cent = gpd.GeoSeries([sacq_5070.geometry.iloc[i].centroid], crs=5070).to_crs(3857).iloc[0]
            if pid not in seen_sacq:
                seen_sacq[pid] = dict(
                    geometry=sacq_3857.geometry.iloc[i], label=f'SA{rank}',
                    label_x=cent.x, label_y=cent.y,
                    ownership=row['nf_ownership'], acres=row['nf_acres'],
                    sub_patch=row['sub_patch_id'], proposals=str(rank),
                )
            else:
                seen_sacq[pid]['label']     += f'/SA{rank}'
                seen_sacq[pid]['proposals'] += f', {rank}'

        seen_srel = {}
        for i, (rank, row) in enumerate(plot_sub.iterrows()):
            pid  = row['release_parcel_id']
            cent = gpd.GeoSeries([srel_5070.geometry.iloc[i].centroid], crs=5070).to_crs(3857).iloc[0]
            if pid not in seen_srel:
                seen_srel[pid] = dict(
                    geometry=srel_3857.geometry.iloc[i], label=f'SR{rank}',
                    label_x=cent.x, label_y=cent.y,
                    sub_patch=row['sub_patch_id'], acres=row['fed_acres'], proposals=str(rank),
                )
            else:
                seen_srel[pid]['label']     += f'/SR{rank}'
                seen_srel[pid]['proposals'] += f', {rank}'

        sub_acq_unique = gpd.GeoDataFrame(list(seen_sacq.values()), geometry='geometry', crs='EPSG:3857')
        sub_rel_unique = gpd.GeoDataFrame(list(seen_srel.values()), geometry='geometry', crs='EPSG:3857')

    ### base layers
    tiles = gvts.CartoLight.opts(width=950, height=680)

    base = gv.Polygons(parcels_3857, crs=ccrs.GOOGLE_MERCATOR).opts(
        line_color='gray', line_width=0.3, fill_alpha=0,
    )

    patch_layer = gv.Polygons(
        patches_3857,
        vdims=['contig_parcel_id', 'area_acres', 'n_parcels', 'interior_edge_ratio'],
        crs=ccrs.GOOGLE_MERCATOR,
    ).opts(color=PATCH_COLOR, line_color='black', line_width=0.6, fill_alpha=0.65, tools=['hover'])

    acq_layer = gv.Polygons(
        acq_unique[['label', 'ownership', 'acres', 'proposals', 'geometry']],
        vdims=['label', 'ownership', 'acres', 'proposals'],
        crs=ccrs.GOOGLE_MERCATOR,
    ).opts(color=ACQUIRE_COLOR, line_color='black', line_width=1.5, fill_alpha=0.9, tools=['hover'])

    rel_layer = gv.Polygons(
        rel_unique[['label', 'release_patch', 'acres', 'proposals', 'geometry']],
        vdims=['label', 'release_patch', 'acres', 'proposals'],
        crs=ccrs.GOOGLE_MERCATOR,
    ).opts(color=RELEASE_COLOR, line_color='black', line_width=1.5, fill_alpha=0.9, tools=['hover'])

    _label_opts = dict(
        text_font_size='8pt', text_color='white',
        text_font_style='bold', text_align='center', text_baseline='middle',
    )

    acq_labels = hv.Overlay([
        gv.Text(row['label_x'], row['label_y'], row['label'], crs=ccrs.GOOGLE_MERCATOR).opts(**_label_opts)
        for _, row in acq_unique.iterrows()
    ])
    rel_labels = hv.Overlay([
        gv.Text(row['label_x'], row['label_y'], row['label'], crs=ccrs.GOOGLE_MERCATOR).opts(**_label_opts)
        for _, row in rel_unique.iterrows()
    ])

    ### compose map
    swap_map = tiles * base * patch_layer * acq_layer * rel_layer * acq_labels * rel_labels

    if _has_subpatch:
        sub_acq_layer = gv.Polygons(
            sub_acq_unique[['label', 'ownership', 'acres', 'sub_patch', 'proposals', 'geometry']],
            vdims=['label', 'ownership', 'acres', 'sub_patch', 'proposals'],
            crs=ccrs.GOOGLE_MERCATOR,
        ).opts(color=SUB_ACQ_COLOR, line_color='black', line_width=1.5, fill_alpha=0.9, tools=['hover'])

        sub_rel_layer = gv.Polygons(
            sub_rel_unique[['label', 'sub_patch', 'acres', 'proposals', 'geometry']],
            vdims=['label', 'sub_patch', 'acres', 'proposals'],
            crs=ccrs.GOOGLE_MERCATOR,
        ).opts(color=SUB_REL_COLOR, line_color='black', line_width=1.5, fill_alpha=0.9, tools=['hover'])

        sub_acq_labels = hv.Overlay([
            gv.Text(row['label_x'], row['label_y'], row['label'], crs=ccrs.GOOGLE_MERCATOR).opts(**_label_opts)
            for _, row in sub_acq_unique.iterrows()
        ])
        sub_rel_labels = hv.Overlay([
            gv.Text(row['label_x'], row['label_y'], row['label'], crs=ccrs.GOOGLE_MERCATOR).opts(**_label_opts)
            for _, row in sub_rel_unique.iterrows()
        ])
        swap_map = swap_map * sub_acq_layer * sub_rel_layer * sub_acq_labels * sub_rel_labels

    swap_map = swap_map.opts(title='Proposed Land Swaps — Ranked by Interior Edge Ratio Gain')

    ### render to Bokeh and patch fill colors (hv.render() overrides color= via cycle)
    bokeh_fig = hv.render(swap_map)

    _mp_rs = [r for r in bokeh_fig.renderers
              if isinstance(r, GlyphRenderer) and isinstance(r.glyph, _MP)]

    ### _mp_rs order: base(alpha=0), patch, acq, rel, [sub_acq, sub_rel]
    _colors = [PATCH_COLOR, ACQUIRE_COLOR, RELEASE_COLOR]
    _alphas = [0.65, 0.9, 0.9]
    if _has_subpatch:
        _colors += [SUB_ACQ_COLOR, SUB_REL_COLOR]
        _alphas  += [0.9, 0.9]

    for _r, _c, _a in zip(_mp_rs[1:1 + len(_colors)], _colors, _alphas):
        _r.glyph.fill_color = _c
        _r.glyph.fill_alpha = _a

    ### legend via dummy zero-data renderers
    _empty = ColumnDataSource({'xs': [[]], 'ys': [[]]})
    _r1 = bokeh_fig.patches('xs', 'ys', source=_empty, fill_color=PATCH_COLOR,   fill_alpha=0.65, line_color='black')
    _r2 = bokeh_fig.patches('xs', 'ys', source=_empty, fill_color=ACQUIRE_COLOR, fill_alpha=0.9,  line_color='black')
    _r3 = bokeh_fig.patches('xs', 'ys', source=_empty, fill_color=RELEASE_COLOR, fill_alpha=0.9,  line_color='black')

    legend_items = [
        LegendItem(label='Federal patch',                   renderers=[_r1]),
        LegendItem(label='Acquire (non-federal → federal)', renderers=[_r2]),
        LegendItem(label='Release (federal → non-federal)', renderers=[_r3]),
    ]

    if _has_subpatch:
        _r4 = bokeh_fig.patches('xs', 'ys', source=_empty, fill_color=SUB_ACQ_COLOR, fill_alpha=0.9, line_color='black')
        _r5 = bokeh_fig.patches('xs', 'ys', source=_empty, fill_color=SUB_REL_COLOR, fill_alpha=0.9, line_color='black')
        legend_items += [
            LegendItem(label='Sub-patch acquire',  renderers=[_r4]),
            LegendItem(label='Sub-patch release',  renderers=[_r5]),
        ]

    bokeh_fig.add_layout(Legend(
        items=legend_items, location='top_right', click_policy='hide'
    ), 'right')

    ### display inline using components() which reuses BokehJS already loaded by hv.extension
    _script, _div = components(bokeh_fig)
    display(HTML(_div + _script))

    map_out = os.path.join(figures_parcel_matrix_dir, 'land_swap_proposals_interactive.html')
    with open(map_out, 'w') as _f:
        _f.write(file_html(bokeh_fig, CDN, 'Land Swap Proposals'))
    print(f'Saved: {map_out}')